# 00 — Ambiente del taller, con el modelo en el servidor del Tec

**El modelo corre en un servidor con GPU del Tec**, alcanzable por la VPN. El
backend y la página corren en la laptop de cada quien.

Es la configuración con menos piezas: **nadie levanta Ollama y no hay firewall
que abrir.** A cambio aparece un requisito nuevo, y es el que más tiempo cuesta
si se descubre tarde: **todos necesitan estar conectados a la VPN del Tec**, con
cuenta y cliente instalado.

| parte | quién la corre |
|---|---|
| **Parte 1** | **solo el instructor**, antes de que llegue el salón |
| **Parte 2** | **cada asistente**, en su propia laptop |

Si eres asistente, **salta directo a la Parte 2**.

Hay otras dos versiones de este notebook:

| notebook | cuándo |
|---|---|
| `00_localhost.ipynb` | trabajas solo, todo en tu máquina |
| `00_aula.ipynb` | el modelo en la laptop del instructor |
| **`00_tec.ipynb`** | **este**: el modelo en el servidor del Tec |

## El ayudante — lo corren los dos

In [5]:
import subprocess
import sys
from pathlib import Path


def raiz_del_repo():
    """
    La raiz del repo, sin importar desde donde se lanzo Jupyter.

    Puede ser notebooks/ o la raiz, segun como se abrio. Buscamos hacia arriba
    el directorio que tenga infra/ y .env.example.
    """
    p = Path.cwd().resolve()
    for candidato in (p, *p.parents):
        if (candidato / "infra").is_dir() and (candidato / ".env.example").is_file():
            return candidato
    raise RuntimeError(f"No encontre la raiz del repo desde {p}")


RAIZ = raiz_del_repo()
print("repo:", RAIZ)


def correr(comando, cwd=None, titulo=None):
    """Corre un comando e imprime su salida en vivo. Devuelve el codigo de salida."""
    if titulo:
        print(f"--- {titulo} ---")
    proceso = subprocess.Popen(
        comando,
        cwd=str(cwd or RAIZ),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,   # docker manda el progreso a stderr
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    for linea in proceso.stdout:
        print(linea.rstrip())
    codigo = proceso.wait()
    print(f"[codigo de salida: {codigo}]")
    return codigo

repo: <HOME>\Documents\desarrollo\tec\agenticai


---

# Parte 1 — Solo el instructor, antes de que llegue el salón

No hay nada que levantar: el servidor ya corre. Lo que hay que hacer es
**comprobar que responde y que tiene el modelo**, antes de que veinte personas
dependan de eso.

## 1.1 La configuración que vas a dictar

La celda siguiente revisa el `.env`. Es la misma que corre el salón en la Parte
2, y la que define a qué servidor se habla.

In [7]:
# El .env de la raiz, que es UN solo archivo para todo: lo leen el codigo de
# src/paso*, estos notebooks y el contenedor del backend.
#
# Esta celda no edita nada: compara lo que hay contra lo que este notebook
# necesita, y si no coincide dice exactamente que cambiar.
import shutil

DESTINO_ESPERADO = "tec"
MODELO_ESPERADO = "ollama:qwen3.5:9b"

env = RAIZ / ".env"
if not env.exists():
    shutil.copy(RAIZ / ".env.example", env)
    print(f"Cree {env} desde el ejemplo.")

valores = {}
for linea in env.read_text(encoding="utf-8").splitlines():
    linea = linea.strip()
    if linea and not linea.startswith("#") and "=" in linea:
        k, v = linea.split("=", 1)
        valores[k.strip()] = v.strip()

destino = valores.get("OLLAMA_DESTINO", "")
modelo = valores.get("CHAT_MODEL", "")
url = valores.get(f"OLLAMA_URL_{destino.upper()}", "(sin declarar)")

print(f"  OLLAMA_DESTINO = {destino or '(vacio)'}   ->  {url}")
print(f"  CHAT_MODEL     = {modelo or '(vacio)'}")

faltan = []
if destino != DESTINO_ESPERADO:
    faltan.append(f"OLLAMA_DESTINO={DESTINO_ESPERADO}")
if modelo != MODELO_ESPERADO:
    faltan.append(f"CHAT_MODEL={MODELO_ESPERADO}")

if faltan:
    print(f"\n  ESTE NOTEBOOK NECESITA otra configuracion. Edita el .env de la raiz:")
    for f in faltan:
        print(f"      {f}")
    print(f"  y vuelve a correr esta celda.")
else:
    print(f"\n  Configuracion correcta para este notebook.")

  OLLAMA_DESTINO = tec   ->  http://10.49.x.x:11434
  CHAT_MODEL     = ollama:qwen3.5:9b

  Configuracion correcta para este notebook.


## 1.2 El servidor responde y tiene el modelo

La celda pregunta la versión de Ollama y lista los modelos descargados.
**Comprueba que el modelo del `.env` esté en la lista**: si no está, el taller no
corre y hay que pedirlo a quien administra el servidor.

> Conéctate a la VPN antes de correrla. Si falla la conexión, no es el notebook:
> es la VPN o el servidor.

In [8]:
# El servidor del Tec, antes de que llegue nadie.
import requests

sys.path.insert(0, str(RAIZ / "src"))

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno. cargar_env() si, y
# avisa de las claves que borro por ya no estar en el archivo.
from agentlib import cargar_env

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno, ya no estan en el .env: {', '.join(_viejas)}")
from agentlib.ollama import base_url   # resuelve segun OLLAMA_DESTINO

URL = base_url()
print("  servidor:", URL)

v = requests.get(f"{URL}/api/version", timeout=20).json()["version"]
modelos = [m["name"] for m in requests.get(f"{URL}/api/tags", timeout=25).json()["models"]]
print("  ollama  :", v)
print("  modelos :")
for m in sorted(modelos):
    print(f"      {m}")

esperado = MODELO_ESPERADO.split(":", 1)[1]
print()
if esperado in modelos:
    print(f"  '{esperado}' esta descargado. Listo.")
else:
    print(f"  FALTA '{esperado}' en el servidor. Sin el, el taller no corre.")
    print(f"  Pidelo a quien administra el servidor.")

  .env   : <HOME>\Documents\desarrollo\tec\agenticai\.env  (19 claves)
  servidor: http://10.49.x.x:11434
  ollama  : 0.40.13
  modelos :
      nomic-embed-text:latest
      qwen3-coder-next:latest
      qwen3-coder:30b
      qwen3.5:0.8b
      qwen3.5:2b
      qwen3.5:4b
      qwen3.5:9b
      qwen3.6:35b
      qwen3.8:latest

  'qwen3.5:9b' esta descargado. Listo.


## 1.3 Compruébalo desde OTRA laptop

Desde la tuya siempre funciona, y eso no prueba nada sobre lo que alcanzan los
demás. Pídele a quien llegue primero que se conecte a la VPN y corra:

```
curl http://<la-direccion-del-servidor>:11434/api/tags
```

La dirección es la que tengas en `OLLAMA_URL_TEC` en el `.env`. La celda 1.2 de
arriba la imprime, así que puedes copiarla de ahí.

**Es el único paso de esta parte que no se puede automatizar**, y es el que
descubre los problemas de red a tiempo.

---

# Parte 2 — Cada asistente, en su laptop

## 2.1 Conéctate a la VPN del Tec

Sin VPN no hay ruta al servidor, y todo lo demás falla con un error de conexión
que no menciona la VPN por ningún lado. **Si no tienes cuenta o cliente, resuélvelo
ahora**, no cuando empiece el bloque 2.

## 2.2 La configuración

La celda revisa el `.env` de la raíz y, si no coincide con lo que necesita este
notebook, dice qué líneas cambiar. Son dos: el destino y el modelo.

In [10]:
# El .env de la raiz, que es UN solo archivo para todo: lo leen el codigo de
# src/paso*, estos notebooks y el contenedor del backend.
#
# Esta celda no edita nada: compara lo que hay contra lo que este notebook
# necesita, y si no coincide dice exactamente que cambiar.
import shutil

DESTINO_ESPERADO = "tec"
MODELO_ESPERADO = "ollama:qwen3.5:9b"

env = RAIZ / ".env"
if not env.exists():
    shutil.copy(RAIZ / ".env.example", env)
    print(f"Cree {env} desde el ejemplo.")

valores = {}
for linea in env.read_text(encoding="utf-8").splitlines():
    linea = linea.strip()
    if linea and not linea.startswith("#") and "=" in linea:
        k, v = linea.split("=", 1)
        valores[k.strip()] = v.strip()

destino = valores.get("OLLAMA_DESTINO", "")
modelo = valores.get("CHAT_MODEL", "")
url = valores.get(f"OLLAMA_URL_{destino.upper()}", "(sin declarar)")

print(f"  OLLAMA_DESTINO = {destino or '(vacio)'}   ->  {url}")
print(f"  CHAT_MODEL     = {modelo or '(vacio)'}")

faltan = []
if destino != DESTINO_ESPERADO:
    faltan.append(f"OLLAMA_DESTINO={DESTINO_ESPERADO}")
if modelo != MODELO_ESPERADO:
    faltan.append(f"CHAT_MODEL={MODELO_ESPERADO}")

if faltan:
    print(f"\n  ESTE NOTEBOOK NECESITA otra configuracion. Edita el .env de la raiz:")
    for f in faltan:
        print(f"      {f}")
    print(f"  y vuelve a correr esta celda.")
else:
    print(f"\n  Configuracion correcta para este notebook.")

  OLLAMA_DESTINO = tec   ->  http://10.49.x.x:11434
  CHAT_MODEL     = ollama:qwen3.5:9b

  Configuracion correcta para este notebook.


### Y el inventario, desde la semilla

`datos/inventario.csv` es el **estado vivo** del dominio: el backend lo reescribe
en cada `PUT`, así que no viaja en git. En un clon nuevo no existe, y si quedó
sucio de una corrida anterior, los notebooks del 02 al 06 arrancan con números
que no son los que su texto describe.

La celda siguiente lo rehace desde
[`datos/inventario.semilla.csv`](../datos/inventario.semilla.csv) **pisando lo
que hubiera**, e imprime con qué quedó. Es Python puro: no necesita Docker ni el
backend arriba, y en eso se diferencia de
`scripts/<tu-sistema>/reiniciar-datos`, que lo pide por HTTP.

> La carpeta `datos/` está montada dentro del contenedor del backend, así que
> este archivo y el que lee el backend son el mismo.

In [ ]:
# El estado vivo, rehecho desde la semilla. En un clon nuevo no existe, y si
# quedo de una corrida anterior trae los numeros de esa corrida.
import csv
import shutil

SEMILLA = RAIZ / "datos" / "inventario.semilla.csv"
VIVO = RAIZ / "datos" / "inventario.csv"

antes = f"{VIVO.stat().st_size} bytes" if VIVO.exists() else "no existia"
shutil.copy(SEMILLA, VIVO)
print(f"  copiado: {SEMILLA.name}  ->  {VIVO.name}   (antes: {antes})")
print()

with open(VIVO, encoding="utf-8", newline="") as f:
    filas = list(csv.DictReader(f))

print(f"  {'SKU':<15} {'producto':<30} {'exist.':>7} {'precio':>7}")
print(f"  {'-' * 15} {'-' * 30} {'-' * 7} {'-' * 7}")
for r in filas:
    unidades = int(r["existencias"])
    aviso = "  <- en cero" if unidades == 0 else ""
    print(f"  {r['sku']:<15} {r['nombre']:<30} {unidades:>7} "
          f"{float(r['precio']):>7.0f}{aviso}")

print(f"\n  {len(filas)} productos. Con esto arrancan los notebooks 02 al 06.")

## 2.3 Levantar el backend y la página

Corren en tu propia laptop. Sin GPU y sin modelo: son imágenes chicas.

**La primera vez tarda unos minutos.**

In [ ]:
correr(["docker", "compose", "up", "-d", "--build"],
       cwd=RAIZ / "infra" / "taller", titulo="levantando backend y frontend")

## 2.4 Verificar

El diagnóstico completo. Fíjate en la línea de Ollama: tiene que decir la
dirección del servidor del Tec y `(por OLLAMA_DESTINO=tec)`.

In [ ]:
correr([sys.executable, str(RAIZ / "scripts" / "verificar_ambiente.py")])

## 2.5 La primera llamada al modelo

Si devuelve texto, el camino completo funciona: tu laptop, la VPN y el servidor.

Con esta configuración las respuestas son rápidas —**décimas de segundo**, no
los ocho segundos de un modelo en una laptop—, porque el servidor tiene una GPU
grande y el modelo es chico.

In [ ]:
sys.path.insert(0, str(RAIZ / "src"))
from agentlib import chat, ultima_medicion   # noqa: E402

respuesta = chat([{"role": "user", "content": "Responde solo con la palabra LISTO, sin nada mas."}])
print(respuesta["content"])
print()
print("  tokens enviados:", ultima_medicion()["tokens_enviados"])
print("  segundos       :", ultima_medicion()["segundos"])

## 2.6 El backend

In [ ]:
import requests

print("salud  :", requests.get("http://localhost:8000/api/health", timeout=5).json())
print("modelos:")
for m in requests.get("http://localhost:8000/api/modelos", timeout=15).json()["modelos"]:
    print(f"  {'listo' if m['listo'] else '  -  '}  {m['id']}")

Y la página: **http://localhost:4200**

Déjala abierta en otra ventana. Desde el paso 3 en adelante es donde se ve el
efecto de lo que hace el agente.

---

# Al terminar

Solo hay que bajar lo tuyo. **No hay Ollama que bajar ni firewall que cerrar:**
el modelo no corría aquí.

In [3]:
correr(["docker", "compose", "down"],
       cwd=RAIZ / "infra" / "taller", titulo="bajando backend y frontend")

--- bajando backend y frontend ---
 Container agenticai_frontend  Stopping
 Container agenticai_frontend  Stopped
 Container agenticai_frontend  Removing
 Container agenticai_frontend  Removed
 Container agenticai_backend  Stopping
 Container agenticai_backend  Stopped
 Container agenticai_backend  Removing
 Container agenticai_backend  Removed
 Network taller_default  Removing
 Network taller_default  Removed
[codigo de salida: 0]


0

In [4]:
# Que no quedo nada arriba.
correr(["docker", "ps", "--filter", "name=agenticai",
        "--format", "{{.Names}}  {{.Status}}"], titulo="contenedores que siguen vivos")
print("(si no lista ninguno, esta todo abajo)")

--- contenedores que siguen vivos ---
[codigo de salida: 0]
(si no lista ninguno, esta todo abajo)
